# 10 — Feature Engineering + Model Comparison

Starting from **M4b** (best config in nb09, R²=0.356), we add engineered features and compare model families.

| Model | Description |
|-------|-------------|
| M4b_OLS | Baseline linear regression (nb09) |
| M4b_FE_OLS | + style_distance, pre_minutes |
| M4b_FE_Ridge | Ridge regression (alpha via CV) |
| M4b_FE_Lasso | Lasso (alpha via CV) |
| M4b_FE_XGB | XGBoost (tuned via CV) |

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/Users/jorgepadilla/Documents/Documents - Jorge\u2019s MacBook Air/thesis_data'
V1 = f'{DATA_ROOT}/processed_data/model_dataset/v_1'
raw = pd.read_parquet(f'{V1}/model_df.parquet')
print(f'Loaded: {raw.shape}')

Loaded: (7366, 92)


## 1. Data Prep

In [3]:
# Drop GKs + NaN team rows
df = raw[raw['position'] != 'Goalkeeper'].copy()
team_cols = [c for c in df.columns if c.startswith('from_q_proj_') or c.startswith('to_q_') or c.startswith('delta_team_')]
df = df[~df[team_cols].isna().any(axis=1)].copy()
print(f'{len(df):,} transfers (no GKs, no NaN team)')

ALL_PLAYER_QUALITIES = [
    'Active defence', 'Aerial threat', 'Box threat', 'Chance prevention',
    'Composure', 'Defensive heading', 'Dribbling', 'Effectiveness',
    'Finishing', 'Hold-up play', 'Intelligent defence', 'Involvement',
    'Passing quality', 'Poaching', 'Pressing', 'Progression',
    'Providing teammates', 'Run quality', 'Territorial dominance', 'Winning duels',
]
TEAM_ALL = ['DEFENCE', 'DEFENSIVE_TRANSITION', 'ATTACKING_TRANSITION',
            'ATTACK', 'PENETRATION', 'CHANCE_CREATION', 'OUTCOME']

POSITIONS = sorted(df['position'].unique())

# Per-position available qualities
position_qualities = {}
for pos in POSITIONS:
    mask = df['position'] == pos
    available = [q for q in ALL_PLAYER_QUALITIES if df.loc[mask, f'pre_{q}'].isna().mean() < 0.50]
    position_qualities[pos] = available

# Train/test split
train_mask = df['to_season'].isin([2020, 2021, 2022, 2023])
test_mask = df['to_season'] == 2024
print(f'Train: {train_mask.sum():,} | Test: {test_mask.sum():,}')

for pos in POSITIONS:
    excluded = set(ALL_PLAYER_QUALITIES) - set(position_qualities[pos])
    print(f'  {pos:20s}: {len(position_qualities[pos])} qualities (excl: {excluded if excluded else "none"})')

6,677 transfers (no GKs, no NaN team)
Train: 5,387 | Test: 1,290
  Central Defender    : 20 qualities (excl: none)
  Full Back           : 20 qualities (excl: none)
  Midfielder          : 17 qualities (excl: {'Chance prevention', 'Poaching', 'Territorial dominance'})
  Striker             : 18 qualities (excl: {'Chance prevention', 'Territorial dominance'})
  Winger              : 18 qualities (excl: {'Chance prevention', 'Territorial dominance'})


## 2. Feature Engineering

Two new features on top of M4b:
- **style_distance**: euclidean distance between from\_team\_proj and to\_team\_curr vectors (7 qualities) — magnitude of contextual shock
- **pre_minutes**: minutes played at origin team — proxy of consolidation/titularity

In [4]:
# Style distance: euclidean between from_team_proj and to_team_curr (7 qualities)
from_cols = [f'from_q_proj_{q}' for q in TEAM_ALL]
to_cols = [f'to_q_{q}' for q in TEAM_ALL]

diff = df[to_cols].values - df[from_cols].values
df['style_distance'] = np.sqrt((diff ** 2).sum(axis=1))

print(f'style_distance: mean={df["style_distance"].mean():.3f}, std={df["style_distance"].std():.3f}')
print(f'pre_minutes:    mean={df["pre_minutes"].mean():.0f}, std={df["pre_minutes"].std():.0f}')

# Quick distribution
fig = make_subplots(rows=1, cols=2, subplot_titles=['Style Distance', 'Pre Minutes'])
fig.add_trace(go.Histogram(x=df['style_distance'], nbinsx=50, marker_color='#636EFA', showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=df['pre_minutes'], nbinsx=50, marker_color='#EF553B', showlegend=False), row=1, col=2)
fig.update_layout(height=300, margin=dict(t=40, b=30, l=40, r=20))
fig.show()

style_distance: mean=1.735, std=1.129
pre_minutes:    mean=1779, std=702


## 3. Model Comparison

Per position: M4b baseline vs M4b + engineered features across 4 model families.

In [5]:
NEW_FEATURES = ['style_distance', 'pre_minutes']

def get_m4b_cols(qualities):
    """Return (X_cols, Y_cols) for M4b config."""
    from_proj = [f'from_q_proj_{q}' for q in TEAM_ALL]
    to_curr = [f'to_q_{q}' for q in TEAM_ALL]
    pre = [f'pre_{q}' for q in qualities]
    post = [f'post_{q}' for q in qualities]
    return from_proj + to_curr + pre, post

def evaluate_model(model, X_train, Y_train, X_test, Y_test, qualities):
    """Fit, predict, return per-quality metrics."""
    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)
    if Y_pred.ndim == 1:
        Y_pred = Y_pred.reshape(-1, 1)
    rows = []
    for i, q in enumerate(qualities):
        rows.append({
            'quality': q,
            'r2': r2_score(Y_test[:, i], Y_pred[:, i]),
            'mae': mean_absolute_error(Y_test[:, i], Y_pred[:, i]),
        })
    return rows

results = []

for pos in POSITIONS:
    qualities = position_qualities[pos]
    x_base, y_cols = get_m4b_cols(qualities)
    x_fe = x_base + NEW_FEATURES

    pos_df = df[df['position'] == pos]

    # Clean data (need all cols non-null)
    all_needed = x_fe + y_cols  # superset
    clean = pos_df[all_needed].dropna()
    train_idx = clean.index[clean.index.isin(df[train_mask].index)]
    test_idx = clean.index[clean.index.isin(df[test_mask].index)]

    if len(train_idx) < 20 or len(test_idx) < 5:
        print(f'  SKIP {pos}: train={len(train_idx)}, test={len(test_idx)}')
        continue

    # Data matrices
    X_train_base = clean.loc[train_idx, x_base].values
    X_test_base = clean.loc[test_idx, x_base].values
    X_train_fe = clean.loc[train_idx, x_fe].values
    X_test_fe = clean.loc[test_idx, x_fe].values
    Y_train = clean.loc[train_idx, y_cols].values
    Y_test = clean.loc[test_idx, y_cols].values

    # Define models
    models = {
        'M4b_OLS': (LinearRegression(), X_train_base, X_test_base),
        'M4b_FE_OLS': (LinearRegression(), X_train_fe, X_test_fe),
        'M4b_FE_Ridge': (MultiOutputRegressor(RidgeCV(alphas=np.logspace(-3, 3, 20))), X_train_fe, X_test_fe),
        'M4b_FE_Lasso': (MultiOutputRegressor(LassoCV(alphas=np.logspace(-3, 1, 20), max_iter=5000)), X_train_fe, X_test_fe),
        'M4b_FE_XGB': (
            MultiOutputRegressor(XGBRegressor(
                n_estimators=200, max_depth=4, learning_rate=0.1,
                subsample=0.8, colsample_bytree=0.8,
                random_state=42, verbosity=0
            )),
            X_train_fe, X_test_fe
        ),
    }

    for model_name, (model, X_tr, X_te) in models.items():
        rows = evaluate_model(model, X_tr, Y_train, X_te, Y_test, qualities)
        for r in rows:
            r['position'] = pos
            r['model'] = model_name
            r['n_train'] = len(train_idx)
            r['n_test'] = len(test_idx)
        results.extend(rows)
    print(f'  {pos}: done ({len(train_idx)} train, {len(test_idx)} test)')

results_df = pd.DataFrame(results)
print(f'\nTotal evaluations: {len(results_df)}')

  SKIP Central Defender: train=316, test=0
  Full Back: done (652 train, 192 test)
  Midfielder: done (1535 train, 383 test)
  Striker: done (576 train, 117 test)
  Winger: done (763 train, 189 test)

Total evaluations: 365


## 4. Results — Mean R² by Position x Model

In [6]:
MODEL_ORDER = ['M4b_OLS', 'M4b_FE_OLS', 'M4b_FE_Ridge', 'M4b_FE_Lasso', 'M4b_FE_XGB']

pivot_r2 = results_df.groupby(['position', 'model'])['r2'].mean().unstack('model')
pivot_r2 = pivot_r2[[c for c in MODEL_ORDER if c in pivot_r2.columns]]

pivot_mae = results_df.groupby(['position', 'model'])['mae'].mean().unstack('model')
pivot_mae = pivot_mae[[c for c in MODEL_ORDER if c in pivot_mae.columns]]

print('MEAN R\u00b2 BY POSITION x MODEL')
print(pivot_r2.round(4).to_string())
print(f'\nOverall mean R\u00b2:')
print(pivot_r2.mean().round(4).to_string())

MEAN R² BY POSITION x MODEL
model       M4b_OLS  M4b_FE_OLS  M4b_FE_Ridge  M4b_FE_Lasso  M4b_FE_XGB
position                                                               
Full Back    0.2731      0.2707        0.2944        0.3008      0.1811
Midfielder   0.4688      0.4680        0.4693        0.4680      0.4278
Striker      0.3727      0.3703        0.3811        0.3883      0.2719
Winger       0.3102      0.3092        0.3193        0.3191      0.2491

Overall mean R²:
model
M4b_OLS         0.3562
M4b_FE_OLS      0.3546
M4b_FE_Ridge    0.3660
M4b_FE_Lasso    0.3691
M4b_FE_XGB      0.2824


In [7]:
# Heatmap: R² by Position x Model
fig = px.imshow(
    pivot_r2.round(3),
    text_auto='.3f',
    color_continuous_scale='RdYlGn',
    zmin=-0.1, zmax=0.6,
    labels=dict(x='Model', y='Position', color='R\u00b2'),
    aspect='auto',
)
fig.update_layout(
    title='Mean R\u00b2 by Position x Model (test set: 2024)',
    height=350, width=700,
    margin=dict(t=50, b=30, l=120, r=30),
)
fig.show()

In [8]:
# Bar chart: overall mean R² per model
mean_r2 = pivot_r2.mean().reset_index()
mean_r2.columns = ['Model', 'R2']
mean_mae = pivot_mae.mean().reset_index()
mean_mae.columns = ['Model', 'MAE']

fig = make_subplots(rows=1, cols=2, subplot_titles=['Mean R\u00b2 (higher is better)', 'Mean MAE (lower is better)'])

colors = ['#636EFA', '#636EFA', '#EF553B', '#FFA15A', '#00CC96']
fig.add_trace(go.Bar(x=mean_r2['Model'], y=mean_r2['R2'], marker_color=colors,
                      text=mean_r2['R2'].round(3), textposition='outside', showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=mean_mae['Model'], y=mean_mae['MAE'], marker_color=colors,
                      text=mean_mae['MAE'].round(3), textposition='outside', showlegend=False), row=1, col=2)

fig.update_layout(height=400, width=900, margin=dict(t=50, b=80, l=40, r=20))
fig.update_xaxes(tickangle=-30)
fig.show()

## 5. R\u00b2 per Quality — Best Model

In [9]:
best_model = pivot_r2.mean().idxmax()
print(f'Best model: {best_model} (mean R\u00b2 = {pivot_r2.mean()[best_model]:.4f})')

best_df = results_df[results_df['model'] == best_model]
pivot_quality = best_df.pivot_table(index='quality', columns='position', values='r2')

fig = px.imshow(
    pivot_quality.round(3),
    text_auto='.3f',
    color_continuous_scale='RdYlGn',
    zmin=-0.3, zmax=0.8,
    labels=dict(x='Position', y='Quality', color='R\u00b2'),
    aspect='auto',
)
fig.update_layout(
    title=f'R\u00b2 per Quality \u2014 {best_model} (test set)',
    height=600, width=700,
    margin=dict(t=50, b=30, l=160, r=30),
)
fig.show()

Best model: M4b_FE_Lasso (mean R² = 0.3691)


## 6. Feature Importance — XGBoost

In [10]:
# Re-fit XGB for best position and extract feature importances
best_pos = pivot_r2['M4b_FE_XGB'].idxmax()
qualities = position_qualities[best_pos]
x_base, y_cols = get_m4b_cols(qualities)
x_fe = x_base + NEW_FEATURES

pos_df = df[df['position'] == best_pos]
clean = pos_df[x_fe + y_cols].dropna()
train_idx = clean.index[clean.index.isin(df[train_mask].index)]

xgb_model = MultiOutputRegressor(XGBRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
))
xgb_model.fit(clean.loc[train_idx, x_fe].values, clean.loc[train_idx, y_cols].values)

# Average feature importance across all target qualities
importances = np.mean([est.feature_importances_ for est in xgb_model.estimators_], axis=0)
feat_imp = pd.Series(importances, index=x_fe).sort_values(ascending=True)

# Shorten names for display
short = feat_imp.copy()
short.index = (
    short.index
    .str.replace('from_q_proj_', 'from_', regex=False)
    .str.replace('to_q_', 'to_', regex=False)
    .str.replace('pre_', 'pre_', regex=False)
)

fig = px.bar(
    x=short.values, y=short.index,
    orientation='h',
    labels={'x': 'Mean Feature Importance', 'y': ''},
    color=short.values,
    color_continuous_scale='Viridis',
)
fig.update_layout(
    title=f'XGBoost Feature Importance \u2014 {best_pos} (avg across targets)',
    height=max(500, len(x_fe) * 20),
    width=700,
    margin=dict(t=50, b=30, l=180, r=30),
    showlegend=False,
    coloraxis_showscale=False,
)
fig.show()

## 7. Conclusions

In [11]:
# Final comparison table
summary = pd.DataFrame({
    'Mean R\u00b2': pivot_r2.mean().round(4),
    'Mean MAE': pivot_mae.mean().round(4),
})
for pos in pivot_r2.index:
    summary[f'R\u00b2 {pos}'] = pivot_r2.loc[pos].round(4)

print('FINAL MODEL COMPARISON')
print('=' * 90)
print(summary.to_string())

# Deltas vs baseline
baseline_r2 = pivot_r2.mean()['M4b_OLS']
print(f'\n\u0394 R\u00b2 vs M4b_OLS baseline ({baseline_r2:.4f}):')
for m in MODEL_ORDER[1:]:
    delta = pivot_r2.mean()[m] - baseline_r2
    print(f'  {m:18s}: {"+" if delta >= 0 else ""}{delta:.4f}')

FINAL MODEL COMPARISON
              Mean R²  Mean MAE  R² Full Back  R² Midfielder  R² Striker  R² Winger
model                                                                              
M4b_OLS        0.3562    0.4055        0.2731         0.4688      0.3727     0.3102
M4b_FE_OLS     0.3546    0.4061        0.2707         0.4680      0.3703     0.3092
M4b_FE_Ridge   0.3660    0.4026        0.2944         0.4693      0.3811     0.3193
M4b_FE_Lasso   0.3691    0.4018        0.3008         0.4680      0.3883     0.3191
M4b_FE_XGB     0.2824    0.4291        0.1811         0.4278      0.2719     0.2491

Δ R² vs M4b_OLS baseline (0.3562):
  M4b_FE_OLS        : -0.0016
  M4b_FE_Ridge      : +0.0098
  M4b_FE_Lasso      : +0.0129
  M4b_FE_XGB        : -0.0737


In [12]:
# Radar chart: R² by position for top 3 models
top_models = pivot_r2.mean().nlargest(3).index.tolist()
positions = pivot_r2.index.tolist()

fig = go.Figure()
colors_radar = ['#00CC96', '#636EFA', '#EF553B']
for i, m in enumerate(top_models):
    vals = pivot_r2.loc[:, m].tolist()
    vals.append(vals[0])  # close the polygon
    fig.add_trace(go.Scatterpolar(
        r=vals,
        theta=positions + [positions[0]],
        name=m,
        line=dict(color=colors_radar[i], width=2),
        fill='toself',
        opacity=0.3,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[-0.05, 0.55])),
    title='Top 3 Models \u2014 R\u00b2 by Position',
    height=450, width=550,
    margin=dict(t=60, b=30, l=80, r=80),
)
fig.show()